# KG1 V195 focal Colab Pro run

Treino curto e controlado para tentar transformar ganhos offline verificados em adapter submetivel. Nao submete no Kaggle automaticamente.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, zipfile, pathlib, shutil, urllib.request, hashlib, sys, subprocess, importlib.util
ROOT = pathlib.Path('/content/kg1_v195')
PACK = pathlib.Path('/content/drive/MyDrive/KG1_NVIDIA_V195/kg1_v195_colab_pack.zip')
PACK_URL = 'https://raw.githubusercontent.com/FELIPEACASTRO/KG1-NVIDIA/claude/competent-shamir/runs/v195_focal_colab_pack_20260503/kg1_v195_colab_pack.zip'
PACK_SHA256 = 'ac82a921977ecb6de20e40cdc060284361d4cc1f7ca66a86a1cc0f69055b55a3'
BASE_ADAPTER = pathlib.Path('/content/drive/MyDrive/KG1_NVIDIA_V195/init_adapter/final')
BASE_ADAPTER_MODEL_SHA256 = '3d16ba908a5c8808624f1abd8fdc2b29f92723f5c874761161c894d7e5759f21'
BASE_ADAPTER_CONFIG_SHA256 = 'e5499f128fde60d32d0595d427e4fe84d8abe6dbde1d80886c970e8184e4b743'
OUT = pathlib.Path('/content/drive/MyDrive/KG1_NVIDIA_V195/output_v195')

def sha256_path(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

PACK.parent.mkdir(parents=True, exist_ok=True)
if not PACK.exists():
    print('Pack not found in Drive; downloading from GitHub...')
    urllib.request.urlretrieve(PACK_URL, PACK)
pack_hash = sha256_path(PACK)
assert pack_hash == PACK_SHA256, f'Pack SHA mismatch: {pack_hash}'
assert PACK.exists(), f'Missing pack: {PACK}'

BASE_ADAPTER.mkdir(parents=True, exist_ok=True)
adapter_model = BASE_ADAPTER / 'adapter_model.safetensors'
adapter_config = BASE_ADAPTER / 'adapter_config.json'
if not adapter_model.exists() or not adapter_config.exists():
    print('Baseline adapter not found in Drive; downloading public Kaggle dataset aaitdads/my-0p86-adapter...')
    if importlib.util.find_spec('kagglehub') is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub'])
    import kagglehub
    kagglehub.dataset_download('aaitdads/my-0p86-adapter', path='adapter_config.json', output_dir=str(BASE_ADAPTER), force_download=False)
    kagglehub.dataset_download('aaitdads/my-0p86-adapter', path='adapter_model.safetensors', output_dir=str(BASE_ADAPTER), force_download=False)
assert adapter_model.exists(), f'Missing {adapter_model}'
assert adapter_config.exists(), f'Missing {adapter_config}'
model_hash = sha256_path(adapter_model)
config_hash = sha256_path(adapter_config)
assert model_hash == BASE_ADAPTER_MODEL_SHA256, f'Baseline adapter_model SHA mismatch: {model_hash}'
assert config_hash == BASE_ADAPTER_CONFIG_SHA256, f'Baseline adapter_config SHA mismatch: {config_hash}'
print('Baseline adapter OK:', BASE_ADAPTER)
shutil.rmtree(ROOT, ignore_errors=True)
ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(PACK) as zf:
    zf.extractall(ROOT)
assert (ROOT / 'data/v195/v195_focal_train.strict.jsonl').exists()
assert (ROOT / 'scripts/hf_job_train_v90.py').exists()
assert (ROOT / 'scripts/kg1_convert_local_training_adapter_to_kaggle_zip.py').exists()
print('Pack extracted to', ROOT)


In [ ]:
%cd /content/kg1_v195
!pip install -q --upgrade pip
!pip install -q torch transformers accelerate peft datasets safetensors huggingface_hub sentencepiece protobuf


In [ ]:
import os, pathlib
os.environ['UPLOAD_TO_HF'] = '0'
os.environ['MODEL_NAME'] = 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'
os.environ['DATA_FILE'] = '/content/kg1_v195/data/v195/v195_focal_train.strict.jsonl'
os.environ['VAL_FILE'] = '/content/kg1_v195/data/v195/v195_focal_val.jsonl'
os.environ['INIT_ADAPTER_DIR'] = str(BASE_ADAPTER)
os.environ['INIT_ADAPTER_LOAD_MODE'] = 'manual'
os.environ['OUTPUT_DIR'] = str(OUT)
os.environ['V195_OUT'] = str(OUT)
os.environ['RUN_ID'] = 'v195-focal-short-aaitdads86'
os.environ['MAX_LENGTH'] = '2048'
os.environ['BATCH_SIZE'] = '16'
os.environ['MICRO_BATCH_SIZE'] = '1'
os.environ['GRADIENT_CHECKPOINTING'] = '1'
os.environ['MAX_STEPS'] = '110'
os.environ['SAVE_EVERY_STEPS'] = '55'
os.environ['EVAL_EVERY_STEPS'] = '25'
os.environ['EVAL_MAX_EXAMPLES'] = '160'
os.environ['LEARNING_RATE'] = '4e-5'
os.environ['FINAL_LEARNING_RATE'] = '1e-5'
os.environ['EXPECTED_TRAIN_SHA256'] = '8a75affddb2176c4ef46973fb7fdf2389007066a1bb96eb4fae0a7d3c9abed2b'
os.environ['EXPECTED_VAL_SHA256'] = 'fe5530f0252cd47992eb983d063e8de90135b5def592d15daac603d985f26cad'
os.environ['MIN_TRAIN_EXAMPLES'] = '3798'
os.environ['MIN_TOKENIZED_TRAIN_EXAMPLES'] = '3700'
os.environ['MIN_VAL_EXAMPLES'] = '720'
os.environ['MIN_TOKENIZED_VAL_EXAMPLES'] = '700'
os.environ['TRAINABLE_LORA_MODULES'] = 'in_proj,out_proj,q_proj,k_proj,v_proj,o_proj'
os.environ['MAX_TRAINABLE_PARAM_RATIO'] = '0.035'
!python scripts/hf_job_train_v90.py


Converter o adapter treinado para layout Kaggle. Isto nao submete no Kaggle.


In [ ]:
!python scripts/kg1_convert_local_training_adapter_to_kaggle_zip.py \
  --source-adapter-dir "$V195_OUT/final_adapter" \
  --output-dir "$V195_OUT/kaggle_layout" \
  --run-id v195-focal-short-aaitdads86


Depois do treino, validar `output_v195/kaggle_layout/zip/v195-focal-short-aaitdads86_adapter_only.zip` localmente antes de qualquer submissao. O submit so deve seguir se superar o baseline em validacao local e passar gate de regressao.
